In [1]:
import numpy as np
import pandas as pd

In [2]:
# Read Dataset
df = pd.read_csv("Datasets/DateFruit_Dataset.csv")
print(df.shape)
df.head(3)

(898, 35)


,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI


In [3]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Class"] = le.fit_transform(df["Class"])

In [4]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [8]:
# Convert Dataset to Tensor - Use long for y values in multi-class classification
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(np.asarray(y_train), dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(np.asarray(y_test), dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
# Build Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            # 2nd Hidden Layer
            nn.Linear(64, 64),
            nn.ReLU(),
            # Output Layer
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

model = ANN()

# Loss & Optimization
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [11]:
# Training NN
epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        running_loss += loss
        
        loss.backward()
        optimizer.step()  # params update

    train_loss = running_loss/len(train_loader)
    print(f"epoch={epoch+1}/{epochs}, loss={train_loss}")

epoch=1/100, loss=1.7424747943878174
epoch=2/100, loss=1.1633375883102417
epoch=3/100, loss=0.7757852673530579
epoch=4/100, loss=0.585517942905426
epoch=5/100, loss=0.46526485681533813
epoch=6/100, loss=0.3978118598461151
epoch=7/100, loss=0.3529805839061737
epoch=8/100, loss=0.32981282472610474
epoch=9/100, loss=0.3005111813545227
epoch=10/100, loss=0.28176358342170715
epoch=11/100, loss=0.2561027407646179
epoch=12/100, loss=0.2405364066362381
epoch=13/100, loss=0.23880541324615479
epoch=14/100, loss=0.20990869402885437
epoch=15/100, loss=0.2065453678369522
epoch=16/100, loss=0.18996171653270721
epoch=17/100, loss=0.18516770005226135
epoch=18/100, loss=0.17505092918872833
epoch=19/100, loss=0.168064683675766
epoch=20/100, loss=0.1594586819410324
epoch=21/100, loss=0.16628557443618774
epoch=22/100, loss=0.15252961218357086
epoch=23/100, loss=0.1505891978740692
epoch=24/100, loss=0.14988188445568085
epoch=25/100, loss=0.14713874459266663
epoch=26/100, loss=0.141299307346344
epoch=27/100

In [12]:
# Evaluate
total = 0
correct = 0
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        max_val, predicted_label = torch.max(outputs, 1)
        correct += (predicted_label == yb).sum().item()
        total += yb.size(0)

print("Correct vals:", correct)
print("Total vals:", total)
print("Accuracy:", correct/total)

Correct vals: 171
Total vals: 180
Accuracy: 0.95
